# NB3 — Build the labelled dataset: normalize, merge, split

This is the notebook that turns two separate corpora into one training set. It does three things,
and the order matters:

1. **Normalize both classes with the exact same function**, so no formatting artifact survives to
   give the label away.
2. **Merge** into one labelled frame with a unified schema.
3. **Split pair-aware** into train/val/test, so no card's articles straddle a boundary.

**Input:** `ha_corpus.parquet` (3,500 human) + `aig_corpus.parquet` (3,601 AI).
**Output:** `dataset.parquet` with a `split` column, ready for NB4 (Vstat) and NB5 (training).

## Why normalization is the whole ballgame here

Every generator ran `normalize_format` on its own output — newlines collapsed to spaces, markdown
stripped. The human corpus never did: it left NB0b with `light_clean` only. The result is the single
most dangerous asymmetry a text-classification dataset can have:

    articles containing a newline:  human 100%  |  ai 0%

A one-line rule `'\n' in text -> human` scores ~99.9%. A model trained on this learns to count
newlines, not to read Arabic, and every downstream number would be a lie. NB2j flagged this in red;
this notebook fixes it.

The fix is to run the **identical** `normalize_format` on both classes, unconditionally. It's
idempotent — re-running it on the already-normalized AI side changes nothing (verified) — so applying
it everywhere buys symmetry by construction rather than by remembering which side needs it.

## Setup and the shared normalizer

This `normalize_format` is byte-identical to the one every generator used. It must stay that way: if
this function and the generators' ever diverge, the two classes get cleaned differently and the
asymmetry creeps back.

In [1]:
import pandas as pd, numpy as np, re, os, glob, hashlib

OUT_DIR   = '/kaggle/working'
SEED      = 42
VAL_FRAC, TEST_FRAC = 0.10, 0.15      # 75 / 10 / 15 train/val/test, split on cards

def find_parquet(preferred, *keywords):
    if os.path.exists(preferred):
        return preferred
    for kw in keywords:
        hits = [p for p in glob.glob('/kaggle/input/**/*.parquet', recursive=True) if kw in p]
        if hits:
            print(f'(resolved {kw} -> {hits[0]})'); return hits[0]
    print('AVAILABLE /kaggle/input parquet files:')
    for p in glob.glob('/kaggle/input/**/*.parquet', recursive=True): print('   ', p)
    raise FileNotFoundError(preferred)

def normalize_format(text):
    # IDENTICAL to the generators' normalizer. Do not let this drift.
    t = str(text)
    t = re.sub(r'\*\*(.+?)\*\*', r'\1', t)                 # **bold**
    t = re.sub(r'__(.+?)__', r'\1', t)                        # __bold__
    t = re.sub(r'(?<!\w)\*(.+?)\*(?!\w)', r'\1', t)         # *italic*
    t = re.sub(r'^#{1,6}\s*', '', t, flags=re.M)              # # headings
    t = re.sub(r'^\s*[-\u2013\u2014>]\s+', '', t, flags=re.M)  # - bullets, > quotes
    t = re.sub(r'^\s*[-*_]{3,}\s*$', '', t, flags=re.M)       # --- rules
    t = re.sub(r'\n+', ' ', t)                                # newlines -> space
    t = re.sub(r'\s{2,}', ' ', t)                             # collapse whitespace
    return t.strip()

print('normalizer ready')

normalizer ready


## Load both corpora and normalize them the same way

I capture the newline gap before and after so the fix is on the record, not assumed. After this cell
both classes should read 0% newlines.

In [2]:
ha = pd.read_parquet(find_parquet('/kaggle/input/notebooks/bahaaqassem/nb0b-select-corpus/ha_corpus.parquet', 'ha-corpus'))
ai = pd.read_parquet(find_parquet('/kaggle/input/notebooks/bahaaqassem/nb2j-merge-aigfd16f97415/aig_corpus.parquet', 'aig-corpus'))

# make sure the human text column is called 'text'
if 'text' not in ha.columns:
    _cand = [c for c in ha.columns
             if ha[c].dtype == object and ha[c].astype(str).str.len().mean() > 200]
    assert _cand, f'no text column in ha: {list(ha.columns)}'
    ha = ha.rename(columns={_cand[0]: 'text'})

def nl_rate(s): return 100 * s.str.contains(chr(10)).mean()
print(f'newlines BEFORE  human {nl_rate(ha["text"]):.0f}%   ai {nl_rate(ai["text"]):.0f}%')

ha['text'] = ha['text'].apply(normalize_format)
ai['text'] = ai['text'].apply(normalize_format)     # idempotent — already normalized, verified

print(f'newlines AFTER   human {nl_rate(ha["text"]):.0f}%   ai {nl_rate(ai["text"]):.0f}%')
assert nl_rate(ha['text']) == 0 and nl_rate(ai['text']) == 0, 'newlines survived normalization'
print(f'loaded: {len(ha)} human, {len(ai)} ai')

newlines BEFORE  human 100%   ai 0%
newlines AFTER   human 0%   ai 0%
loaded: 3500 human, 3601 ai


## Merge into one labelled frame

Unified schema: `text`, `label` (0 human / 1 ai), `pair_id` (the card — my split key), `generator`
('human' for the human side), and `article_id` (a stable unique row id). I keep `word_count` for the
dataset chapter and later stratification checks; everything generator-specific stays on the AI side
only and isn't needed here.

In [3]:
human = pd.DataFrame({
    'article_id': 'HU_' + ha['id'].astype(str),
    'text':       ha['text'],
    'label':      0,
    'pair_id':    ha['id'].astype(str),        # human card id == HA_xxxxx
    'generator':  'human',
})
ai_df = pd.DataFrame({
    'article_id': ai['id'].astype(str),        # already unique: AI_<gen>_HA_xxxxx
    'text':       ai['text'],
    'label':      1,
    'pair_id':    ai['source_pair_id'].astype(str),
    'generator':  ai['generator'].astype(str),
})

data = pd.concat([human, ai_df], ignore_index=True)
data['word_count'] = data['text'].str.split().str.len()

assert data['article_id'].is_unique, 'duplicate article_id'
assert data['text'].str.len().gt(0).all(), 'empty text after normalize'

print('merged:', len(data), 'articles')
print(data['label'].value_counts().rename({0: 'human', 1: 'ai'}).to_string())
print(f'balance: {100*data["label"].mean():.1f}% ai')

merged: 7101 articles
label
ai       3601
human    3500
balance: 50.7% ai


## Pair-aware split — the second thing NB3 must not get wrong

The unit I split on is the **card** (`pair_id`), never the row. A single card can carry three
articles — one human and up to two AI — that describe the same events with the same entities and
figures. If any of those land on opposite sides of a train/test boundary, the model sees test
content during training and the score inflates.

So I assign each *card* to a split, then every article inherits its card's assignment. I hash the
card id for a deterministic, reproducible assignment that doesn't depend on row order.

In [4]:
def bucket(pair_id):
    h = int(hashlib.md5(pair_id.encode()).hexdigest(), 16) % 10000 / 10000
    if h < TEST_FRAC:               return 'test'
    if h < TEST_FRAC + VAL_FRAC:    return 'val'
    return 'train'

card_split = {pid: bucket(pid) for pid in data['pair_id'].unique()}
data['split'] = data['pair_id'].map(card_split)

# verify: no card appears in more than one split
leak = (data.groupby('pair_id')['split'].nunique() > 1).sum()
assert leak == 0, f'{leak} cards leak across splits'
print('cards leaking across splits:', leak, '(must be 0)')

print('\narticles per split:')
print(data['split'].value_counts().to_string())
print('\nlabel balance within each split:')
print(data.groupby('split')['label'].mean().round(3).to_string(), '  (fraction ai)')
print('\ncards per split:')
print(data.groupby('split')['pair_id'].nunique().to_string())

cards leaking across splits: 0 (must be 0)

articles per split:
split
train    5363
test     1093
val       645

label balance within each split:
split
test     0.507
train    0.507
val      0.507   (fraction ai)

cards per split:
split
test      539
train    2643
val       318


## Re-check the cheap-signal baseline AFTER normalization

The number to watch. Before normalization the newline trap made this trivially ~99.9%; after a
correct symmetric clean it should sit back where NB2j measured it on content alone (~65%). A big jump
would mean normalization introduced a *new* asymmetry — so this cell is the proof the fix worked, not
just that it ran.

I fit on train only and score on test, pair-aware, so this baseline is itself leak-free and directly
comparable to what the hybrid model will report.

In [5]:
from sklearn.linear_model import LogisticRegression
QUOTES = '[\u0022\u00ab\u00bb\u201c\u201d]'

def cheap_feats(t):
    return [len(re.findall(r'[0-9\u0660-\u0669]', t)),
            len(re.findall(QUOTES, t)),
            t.count(':'), t.count('('), t.count(chr(10)), len(t.split())]

tr = data[data['split'] == 'train']
te = data[data['split'] == 'test']
Xtr = np.array([cheap_feats(t) for t in tr['text']]); ytr = tr['label'].values
Xte = np.array([cheap_feats(t) for t in te['text']]); yte = te['label'].values

clf = LogisticRegression(max_iter=2000, class_weight='balanced').fit(Xtr, ytr)
acc = clf.score(Xte, yte)
print(f'cheap-signal baseline (train->test, pair-aware): {acc:.1%}')
print('  newline feature is included on purpose — if the fix worked it now carries no signal')
print('  -> ' + ('SAFE: symmetric, no artifact leak' if acc < 0.75 else
                  'INVESTIGATE: normalization left an asymmetry'))

def present(texts, pat): return 100 * np.mean([bool(re.search(pat, t)) for t in texts])
print(f'\n{"signal":<10}{"human":>7}{"ai":>7}{"gap":>7}')
h_txt = data[data['label'] == 0]['text']; a_txt = data[data['label'] == 1]['text']
for name, pat in [('digits', r'[0-9\u0660-\u0669]'), ('quotes', QUOTES),
                  ('colons', ':'), ('parens', r'\('), ('newlines', chr(10))]:
    h, a = present(h_txt, pat), present(a_txt, pat)
    print(f'{name:<10}{h:>7.0f}{a:>7.0f}{abs(h-a):>7.0f}')

cheap-signal baseline (train->test, pair-aware): 68.8%
  newline feature is included on purpose — if the fix worked it now carries no signal
  -> SAFE: symmetric, no artifact leak

signal      human     ai    gap
digits         90     89      1
quotes         92     73     19
colons         60     38     22
parens         69     26     43
newlines        0      0      0


## Save

In [6]:
KEEP = ['article_id', 'text', 'label', 'pair_id', 'generator', 'word_count', 'split']
out = data[KEEP].sample(frac=1.0, random_state=SEED).reset_index(drop=True)   # shuffle rows

OUT = f'{OUT_DIR}/dataset.parquet'
out.to_parquet(OUT, index=False)
print('saved:', OUT, '|', out.shape)
print('\ncolumns:', list(out.columns))
print(f'\n{len(out)} articles | {out["pair_id"].nunique()} cards | '
      f'{100*out["label"].mean():.1f}% ai')
print('splits:', out['split'].value_counts().to_dict())
print('upload as aigt-dataset for NB4')

saved: /kaggle/working/dataset.parquet | (7101, 7)

columns: ['article_id', 'text', 'label', 'pair_id', 'generator', 'word_count', 'split']

7101 articles | 3500 cards | 50.7% ai
splits: {'train': 5363, 'test': 1093, 'val': 645}
upload as aigt-dataset for NB4


## Notes

**Contract for NB4 / NB5 — what this dataset guarantees, and what it demands:**

- **Guaranteed symmetric.** Both classes passed through the identical `normalize_format`; the newline
  gap is gone (100%/0% -> 0%/0%) and the cheap-signal baseline is back at content level. If NB4 or
  NB5 re-reads raw text from anywhere else, it must apply the same normalizer or the symmetry breaks.
- **`split` is pair-aware and final.** Train/val/test are assigned by card hash; every article of a
  card shares one split. Never re-split on row index — 104 cards carry two AI articles plus a human
  article, and a naive split leaks. For k-fold in NB8, fold on `pair_id`, the same way.
- **Fit scalers and the perplexity surrogate on train only.** NB4's statistical features (especially
  anything normalized or standardized) must be fit on the train split and applied to val/test, or the
  Vstat features leak test distribution into training.
- **Label is 0 = human, 1 = ai.** `class_weight='balanced'` handles the 50.7% AI skew; don't subsample.

**For the dataset chapter:**

- Final dataset: 7,101 articles (3,500 human + 3,601 AI) over 3,497 cards, split 75/10/15 by card.
- The symmetric-normalization step is worth describing explicitly as a leakage control — it's the
  kind of methodological care reviewers look for, and the before/after newline numbers make it
  concrete.
- Cheap-signal control baseline (punctuation + length only, train->test) is reported here; the hybrid
  model in NB5-NB8 should be compared against it, not just against chance.